In [0]:
catalog = "workspace"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"   

landing_path = f"/Volumes/{catalog}/default/inputs"

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")



In [0]:
from pyspark.sql.functions import current_timestamp


path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_reviews = f"{landing_path}/movies_reviews.csv"



df_credits_and_tags_raw = spark.read.csv(path_credits_and_tags, header=True, inferSchema=True)
df_financials_raw = spark.read.csv(path_financials, header=True, inferSchema=True)
df_info_raw = spark.read.csv(path_info, header=True, inferSchema=True)
df_metrics_raw = spark.read.csv(path_metrics, header=True, inferSchema=True)
df_reviews_raw = spark.read.csv(path_reviews, header=True, inferSchema=True)


In [0]:
df_credits_and_tags_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_credits_and_tags")

df_financials_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_financials")

df_info_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_info")

df_metrics_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_metrics")

df_reviews_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_reviews")


display(spark.table(f"{catalog}.bronze.tb_movies_reviews").limit(5))

display(spark.table(f"{catalog}.bronze.tb_movies_info").limit(5))

In [0]:
import pandas as pd
import requests
from datetime import datetime, timedelta
from pyspark.sql.functions import current_timestamp
dbutils.widgets.removeAll()
hoje = datetime.now()
print(hoje.strftime("%m-%d-%Y"))

sete_dias_atras = hoje - timedelta(days=7)
print(sete_dias_atras.strftime("%m-%d-%Y"))

data = "%m-%d-%Y"

data_inicio_padrao = sete_dias_atras.strftime(data)
data_fim_padrao = hoje.strftime(data)
print(data_inicio_padrao)
print(data_fim_padrao)

dbutils.widgets.text("data_inicio", data_inicio_padrao,"Data Inicio(MM-DD-YYYY)")
dbutils.widgets.text("data_fim", data_fim_padrao,"Data Fim(MM-DD-YYYY)")

data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

endpoint = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

response = requests.get(endpoint)
if response.status_code == 200:
    cotacoes = response.json()
    if "value" in cotacoes and len(cotacoes['value']) > 0:
        pdf_cotacao = pd.DataFrame(cotacoes['value'])
        df_cotacao = spark.createDataFrame(pdf_cotacao)

        df_cotacao_bronze = df_cotacao.withColumn("ingestion_datetime", current_timestamp())
        df_cotacao_bronze.write.format("delta").mode("append").saveAsTable(f"{catalog}.bronze.tb_cotacao_dolar")
        print("Dados gravados com sucesso!")
    else:
        print("Nenhuma cotação encontrada no período especificado.")
else:
    print(f"Erro ao fazer a requisição: Status Code {response.status_code}")
display(spark.table(f"{catalog}.bronze.tb_cotacao_dolar"))
        

